<a href="https://colab.research.google.com/github/shah833/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shah833/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Random Forest over Gradient Boosting for my model. Gradient Boosting builds trees sequentially, each correcting the previous tree's errors, which makes it more prone to overfitting without careful tuning. Random Forest builds many trees independently and averages their votes, making it a safer, more stable choice for an initial model — and prior exploration (Notebook 1, Week 1) already showed a Random Forest outperforming a hand-written rule by a wide margin on similar data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The data is split by client_id, ensuring every page from a given client appears entirely in either the training set or the test set, never both. A random row-level split would risk leakage: since a single client can have many pages, the model could implicitly learn client-specific patterns during training and then be unfairly evaluated on other pages from that same, now-familiar client — inflating performance in a way that wouldn't hold up on a genuinely new client.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
import os, sys, subprocess

if not os.path.isdir("Flyrank-ML-Internship"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shah833/Flyrank-ML-Internship"])

os.chdir("Flyrank-ML-Internship")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head(10)

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [7]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining"].value_counts())

is_declining
1    16262
0    13738
Name: count, dtype: int64


In [8]:
base_rate = df["is_declining"].value_counts(normalize=True).max()
print(f"Base rate: {base_rate:.1%}")

Base rate: 54.2%


In [9]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct',
 'is_declining']

In [14]:
print(df[features].isna().sum())

avg_position            0
ctr                     0
competition_level    2610
impressions_90d         0
content_type            0
dtype: int64


In [15]:
df["competition_level"] = df["competition_level"].fillna("Unknown")

In [16]:
features = ["avg_position", "ctr", "competition_level", "impressions_90d", "content_type"]

X = df[features].copy()
y = df["is_declining"]

X = pd.get_dummies(X, columns=["content_type", "competition_level"], drop_first=True)
print(X.shape)
X.head()

(30000, 8)


,avg_position,ctr,impressions_90d,content_type_feedly article,content_type_keyword article,competition_level_LOW,competition_level_MEDIUM,competition_level_Unknown
0,10.6,0.76,3803,False,True,False,False,False
1,20.3,0.05,15320,False,True,True,False,False
2,36.5,0.09,12581,False,True,True,False,False
3,6.2,0.49,11751,False,True,True,False,False
4,44.0,0.13,19140,False,True,True,False,False


In [17]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 23837
Test size: 6163


In [19]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [20]:
probs = model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results["true_label"] = y_test.values
results["predicted_prob"] = probs

results_sorted = results.sort_values("predicted_prob", ascending=False)
top50 = results_sorted.head(50)

precision_at_50 = top50["true_label"].mean()
print(f"Model Precision@50: {precision_at_50:.3f}")

Model Precision@50: 0.620


In [21]:
# Get the test set rows from the original dataframe (with all original columns)
test_df = df.loc[X_test.index].copy()

# Reapply your baseline eligibility + rule
eligible_test = test_df[(test_df["impressions_90d"] >= 80) & (test_df["days_since_last_update"] > 60)].copy()
eligible_test["peer_ctr"] = eligible_test.groupby("position_tier")["ctr"].transform("mean")
eligible_test["ctr_gap_pct"] = (eligible_test["peer_ctr"] - eligible_test["ctr"]) / eligible_test["peer_ctr"]
eligible_test["score"] = eligible_test["ctr_gap_pct"]

baseline_sorted = eligible_test.sort_values(["score", "impressions_90d"], ascending=[False, False])
baseline_top50 = baseline_sorted.head(50)

baseline_correct = baseline_top50["is_declining"].sum()
baseline_precision_at_50 = baseline_correct / 50
print(f"Baseline Correct: {baseline_correct} out of 50")
print(f"Baseline Precision@50: {baseline_precision_at_50:.3f}")

Baseline Correct: 22 out of 50
Baseline Precision@50: 0.440


# **4. Erro and interpretation**

In [22]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm_result.importances_mean
}).sort_values("importance", ascending=False)

print(importance_df)

                        feature  importance
2               impressions_90d    0.040922
0                  avg_position    0.018887
1                           ctr    0.001136
7     competition_level_Unknown    0.000990
3   content_type_feedly article    0.000000
4  content_type_keyword article    0.000000
6      competition_level_MEDIUM   -0.000990
5         competition_level_LOW   -0.002158


**Interpretation**

The most useful feature by far was impressions_90d (0.041), followed by avg_position (0.019). Surprisingly, ctr — the one signal the whole baseline rule was built on — barely mattered (0.001). Content type and competition level added almost nothing, and a couple even slightly hurt the model.

This matches something I already noticed while building the baseline: pages with zero CTR all scored the same until I added impressions as a tiebreaker. The model found the same thing on its own — impressions matter more than CTR. This is a useful, honest finding: the baseline may be leaning on the wrong signal, and a future version should probably weigh impressions more.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.